# Transform Your Data

## Introduction to Transforming Data

[**Feature engineering**](https://developers.google.com/machine-learning/glossary#feature_engineering) is the process of determining which features might be useful in training a model, and then creating those features by transforming raw data found in log files and other sources. In this section, we focus on when and how to transform numeric and categorical data, and the tradeoffs of different approaches.

### Reasons for Data Transformation

We transform features primarily for the following reasons:

1. **Mandatory transformations** for data compatibility. Examples include:

  * Converting non-numeric features into numeric. You can’t do matrix multiplication on a string, so we must convert the string to some numeric representation.

  * Resizing inputs to a fixed size. Linear models and feed-forward neural networks have a fixed number of input nodes, so your input data must always have the same size. For example, image models need to reshape the images in their dataset to a fixed size.

2. **Optional quality transformations** that may help the model perform better. Examples include:

  * Tokenization or lower-casing of text features.

  * Normalized numeric features (most models perform better afterwards).
  
  * Allowing linear models to introduce non-linearities into the feature space.

### Where to Transform?

You can apply transformations either while generating the data on disk, or within the model.

**Transforming prior to training**

In this approach, we perform the transformation before training. This code lives separate from your machine learning model.

**Pros**

* Computation is performed only once.
* Computation can look at entire dataset to determine the transformation.

**Cons**

* Transformations need to be reproduced at prediction time. Beware of skew!
* Any transformation changes require rerunning data generation, leading to slower iterations.

Skew is more dangerous for cases involving online serving. In offline serving, you might be able to reuse the code that generates your training data. In online serving, the code that creates your dataset and the code used to handle live traffic are almost necessarily different, which makes it easy to introduce skew.

**Transforming within the model**

In this approach, the transformation is part of the model code. The model takes in untransformed data as input and will transform it within the model.

**Pros**

* Easy iterations. If you change the transformations, you can still use the same data files.
* You're guaranteed the same transformations at training and prediction time.

**Cons**

* Expensive transforms can increase model latency.
* Transformations are per batch.

There are many considerations for transforming per batch. Suppose you want to [**normalize**](https://developers.google.com/machine-learning/glossary#normalization) a feature by its average value--that is, you want to change the feature values to have mean `0` and standard deviation `1`. When transforming inside the model, this normalization will have access to only one batch of data, not the full dataset. You can either normalize by the average value within a batch (dangerous if batches are highly variant), or precompute the average and fix it as a constant in the model.

### Explore, Clean, and Visualize Your Data

Explore and clean up your data before performing any transformations on it. You may have done some of the following tasks as you collected and constructed your dataset:

* Examine several rows of data.
* Check basic statistics.
* Fix missing numerical entries.

**Visualize your data frequently.** Graphs can help find anomalies or patterns that aren't clear from numerical statistics. Therefore, before getting too far into analysis, look at your data graphically, either via scatter plots or histograms. View graphs not only at the beginning of the pipeline, but also throughout transformation. Visualizations will help you continually check your assumptions and see the effects of any major changes.

## Transforming Numeric Data

You may need to apply two kinds of transformations to numeric data:

* **Normalizing** - transforming numeric data to the same scale as other numeric data.
* **Bucketing** - transforming numeric (usually continuous) data to categorical data.

### Why Normalize Numeric Features?

We strongly recommend normalizing a data set that has numeric features covering distinctly different ranges (for example, age and income). When different features have different ranges, gradient descent can "bounce" and slow down convergence. Optimizers like [Adagrad](https://developers.google.com/machine-learning/glossary#adagrad) and [Adam](https://arxiv.org/abs/1412.6980) protect against this problem by creating a separate effective learning rate for each feature.

We also recommend normalizing a single numeric feature that covers a wide range, such as "city population." If you don't normalize the "city population" feature, training the model might generate NaN errors. Unfortunately, optimizers like Adagrad and Adam can't prevent NaN errors when there is a wide range of values within a single feature.

###### Warning: When normalizing, ensure that the same normalizations are applied at serving time to avoid skew.

### Normalization

The goal of normalization is to transform features to be on a similar scale. This improves the performance and training stability of the model.

Four common normalization techniques may be useful:

* scaling to a range
* clipping
* log scaling
* z-score

The following charts show the effect of each normalization technique on the distribution of the raw feature (price) on the left. The charts are based on the data set from 1985 Ward's Automotive Yearbook that is part of the [UCI Machine Learning Repository under Automobile Data Set](https://archive.ics.uci.edu/ml/datasets/automobile).

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/normalizations-at-a-glance-v2.svg" />

<strong>Figure 1. Summary of normalization techniques.</strong>

</div>

#### Scaling to a range

[Recall from MLCC](https://developers.google.com/machine-learning/crash-course/representation/cleaning-data) that [**scaling**](https://developers.google.com/machine-learning/glossary#scaling) means converting floating-point feature values from their natural range (for example, 100 to 900) into a standard range—usually 0 and 1 (or sometimes -1 to +1). Use the following simple formula to scale to a range:

$$x' = \dfrac{(x - x_{min})}{(x_{max} - x_{min})}$$

Scaling to a range is a good choice when both of the following conditions are met:

* You know the approximate upper and lower bounds on your data with few or no outliers.
* Your data is approximately uniformly distributed across that range.

A good example is age. Most age values falls between 0 and 90, and every part of the range has a substantial number of people.

In contrast, you would not use scaling on income, because only a few people have very high incomes. The upper bound of the linear scale for income would be very high, and most people would be squeezed into a small part of the scale.

#### Feature Clipping

If your data set contains extreme outliers, you might try feature clipping, which caps all feature values above (or below) a certain value to fixed value. For example, you could clip all temperature values above 40 to be exactly 40.

You may apply feature clipping before or after other normalizations.

**Formula: Set min/max values to avoid outliers.**

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/norm-clipping-outliers.svg" />

<strong>Figure 2. Comparing a raw distribution and its clipped version.</strong>

</div>

Another simple clipping strategy is to clip by z-score to +-Nσ (for example, limit to +-3σ). Note that σ is the standard deviation.

#### Log Scaling

Log scaling computes the log of your values to compress a wide range to a narrow range.

$$x' = log(x)$$

Log scaling is helpful when a handful of your values have many points, while most other values have few points. This data distribution is known as the *power law* distribution. Movie ratings are a good example. In the chart below, most movies have very few ratings (the data in the tail), while a few have lots of ratings (the data in the head). Log scaling changes the distribution, helping to improve linear model performance.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/norm-log-scaling-movie-ratings.svg" />

<strong>Figure 3. Comparing a raw distribution to its log.</strong>

</div>

#### Z-Score

Z-score is a variation of scaling that represents the number of standard deviations away from the mean. You would use z-score to ensure your feature distributions have mean = 0 and std = 1. It’s useful when there are a few outliers, but not so extreme that you need clipping.

The formula for calculating the z-score of a point, *x*, is as follows:

$$x' = \dfrac{x - μ}{σ}$$

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/norm-z-score.svg" />

<strong>Figure 4. Comparing a raw distribution to its z-score distribution.</strong>

</div>

Notice that z-score squeezes raw values that have a range of ~40000 down into a range from roughly -1 to +4.

Suppose you're not sure whether the outliers truly are extreme. In this case, start with z-score unless you have feature values that you don't want the model to learn; for example, the values are the result of measurement error or a quirk.

### Bucketing

Let's revisit the latitude plot from above.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/ScalingBinningPart1.svg" />

<strong>Figure 1: House prices versus latitude.</strong>

</div>

In cases like the latitude example when there's no linear relationship between the feature and the result, you need to divide the latitudes into buckets to learn something different about housing values for each bucket. This transformation of numeric features into categorical features, using a set of thresholds, is called [**bucketing**](https://developers.google.com/machine-learning/glossary#bucketing) (or binning). In this bucketing example, the boundaries are equally spaced.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/ScalingBinningPart2.svg" />

<strong>Figure 2: House prices versus latitude, now divided into buckets.</strong>

</div>

### Quantile Bucketing

With one feature per bucket, the model uses as much capacity for a single example in the >45000 range as for all the examples in the 5000-10000 range. This seems wasteful. How might we improve this situation?

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/bucketizing-needed.svg" />

<strong>Figure 3: Number of cars sold at different prices.</strong>

</div>

The problem is that equally spaced buckets don’t capture this distribution well. The solution lies in creating buckets that each have the same number of points. This technique is called [**quantile bucketing**](https://developers.google.com/machine-learning/glossary#quantile_bucketing). For example, the following figure divides car prices into quantile buckets. In order to get the same number of examples in each bucket, some of the buckets encompass a narrow price span while others encompass a very wide price span.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/bucketizing-applied.svg" />

<strong>Figure 4: Quantile bucketing gives each bucket about the same number of cars.</strong>

</div>

### Summary

If you choose to bucketize your numerical features, be clear about how you are setting the boundaries and which type of bucketing you’re applying:

* **Buckets with equally spaced boundaries:** the boundaries are fixed and encompass the same range (for example, 0-4 degrees, 5-9 degrees, and 10-14 degrees, or $5,000-$9,999, $10,000-$14,999, and $15,000-$19,999). Some buckets could contain many points, while others could have few or none.
* **Buckets with quantile boundaries:** each bucket has the same number of points. The boundaries are not fixed and could encompass a narrow or wide span of values.

## Transforming Categorical Data

Some of your features may be discrete values that aren’t in an ordered relationship. Examples include breeds of dogs, words, or postal codes. These features are known as [**categorical**](https://developers.google.com/machine-learning/glossary#categorical_data) and each value is called a category. You can represent categorical values as strings or even numbers, but you won't be able to compare these numbers or subtract them from each other.

Oftentimes, you should represent features that contain integer values as categorical data instead of as numerical data. For example, consider a postal code feature in which the values are integers. If you mistakenly represent this feature numerically, then you're asking the model to find a numeric relationship between different postal codes; for example, you're expecting the model to determine that postal code 20004 is twice (or half) the signal as postal code 10002. By representing postal codes as categorical data, you enable the model to find separate signals for each individual postal code.

If the number of categories of a data field is small, such as the day of the week or a limited palette of colors, you can make a unique feature for each category. For example:

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/categorical-netview.svg" />

<strong>Figure 1: A unique feature for each category.</strong>

</div>

A model can then learn a separate weight for each color. For example, perhaps the model could learn that red cars are more expensive than green cars.

The features can then be indexed.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/categorical-netview-indexed.svg" />

<strong>Figure 2: Indexed features.</strong>

</div>

This sort of mapping is called a **vocabulary**.

### Vocabulary

In a vocabulary, each value represents a unique feature.

| Index Number | Category |
|--|--|
| 0 | Red |
| 1 | Orange |
| 2 | Blue |
| ... | ... |

The model looks up the index from the string, assigning 1.0 to the corresponding slot in the feature vector and 0.0 to all the other slots in the feature vector.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/vocabulary-index-sparse-feature.svg" />

<strong>Figure 3: The end-to-end process to map categories to feature vectors.</strong>

</div>

**Note about sparse representation**

If your categories are the days of the week, you might, for example, end up representing Friday with the feature vector [0, 0, 0, 0, 1, 0, 0]. However, most implementations of ML systems will represent this vector in memory with a sparse representation. A common representation is a list of non-empty values and their corresponding indices—for example, 1.0 for the value and [4] for the index. This allows you to spend less memory storing a huge amount of 0s and allows more efficient matrix multiplication. In terms of the underlying math, the [4] is equivalent to [0, 0, 0, 0, 1, 0, 0].

### Out of Vocab (OOV)

Just as numerical data contains outliers, categorical data does, as well. For example, consider a data set containing descriptions of cars. One of the features of this data set could be the car's color. Suppose the common car colors (black, white, gray, and so on) are well represented in this data set and you make each one of them into a category so you can learn how these different colors affect value. However, suppose this data set contains a small number of cars with eccentric colors (mauve, puce, avocado). Rather than giving each of these colors a separate category, you could lump them into a catch-all category called **Out of Vocab (OOV)**. By using OOV, the system won't waste time training on each of those rare colors.

#### Hashing

Another option is to hash every string (category) into your available index space. Hashing often causes collisions, but you rely on the model learning some shared representation of the categories in the same index that works well for the given problem.

For important terms, hashing can be worse than selecting a vocabulary, because of collisions. On the other hand, hashing doesn't require you to assemble a vocabulary, which is advantageous if the feature distribution changes heavily over time.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/vocab-hash-string.svg" />

<strong>Figure 4: Mapping items to a vocabulary.</strong>

</div>

#### Hybrid of Hashing and Vocabulary

You can take a hybrid approach and combine hashing with a vocabulary. Use a vocabulary for the most important categories in your data, but replace the OOV bucket with multiple OOV buckets, and use hashing to assign categories to buckets.

The categories in the hash buckets must share an index, and the model likely won't make good predictions, but we have allocated some amount of memory to attempt to learn the categories outside of our vocabulary.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/vocab-hybrid.svg" />

<strong>Figure 5: Hybrid approach combining vocabulary and hashing.</strong>

</div>

### Note about Embeddings

An [**embedding**](https://developers.google.com/machine-learning/crash-course/glossary#embeddings) is a categorical feature represented as a continuous-valued feature. Deep models frequently convert the indices from an index to an embedding.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/data-prep/images/vocabulary-index-sparse-feature-embedding.svg" />

<strong>Figure 6: Sparse feature vectors via embedding</strong>

</div>

The other transformations we've discussed could be stored on disk, but embeddings are different. Since embeddings are trained, they're not a typical data transformation—they are part of the model. They're trained with other model weights, and functionally are equivalent to a layer of weights.

What about pretrained embeddings? Pretrained embeddings are still typically modifiable during training, so they're still conceptually part of the model.

# Regularization for Simplicity

**Regularization** means penalizing the complexity of a model to reduce overfitting.

## L2 Regularization

Consider the following **generalization curve**, which shows the loss for both the training set and validation set against the number of training iterations.

<div align='center'>

  <img src='https://developers.google.com/static/machine-learning/crash-course/images/RegularizationTwoLossFunctions.svg' />

  <strong>Figure 1. Loss on training set and validation set.</strong>
</div>

Figure 1 shows a model in which training loss gradually decreases, but validation loss eventually goes up. In other words, this generalization curve shows that the model is [overfitting](https://developers.google.com/machine-learning/crash-course/generalization/peril-of-overfitting) to the data in the training set. Channeling our inner [Ockham](https://developers.google.com/machine-learning/crash-course/generalization/peril-of-overfitting#ockham), perhaps we could prevent overfitting by penalizing complex models, a principle called **regularization**.

In other words, instead of simply aiming to minimize loss (empirical risk minimization):

<div class="math-jax-block">

$$\mathrm{minimize(Loss(Data \vert Model))}$$

</div>

we'll now minimize loss+complexity, which is called **structural risk minimization**.

<div class="math-jax-block">

$$\mathrm{minimize(Loss(Data \vert Model) + \mathrm{complexity(Model)})}$$

</div>

Our training optimization algorithm is now a function of two terms: the **loss term**, which measures how well the model fits the data, and the **regularization term**, which measures model complexity.

There are two commom ways to think of model complexity:

  * Model complexity as a function of the *weights* of all the features in the model.
  * Model complexity as a function of the *total number of features* with nonzero weights.

If model complexity is a function of weights, a feature weight with a high absolute value is more complex that a feature weight with a low absolute value.

We can quantify complexity using the $L_{2}$ **regularization** formula, which defines the regularization term as the sum of the squares of all the feature weights:

<div class="math-jax-block">

$$\mathrm{L_{2}\ regularization\ term} = \mathrm{\lvert \lvert w \rvert \rvert}^2_{2} = w_{1}^2 + w_{2}^2 + ... + w_{n}^2$$

</div>

In this formula, weights close to zero have little effect on model complexity, while outlier weights can have a huge impact.

For example, a linear model with the following weights:

<div class="math-jax-block">

$$\{w_{1} = 0.2, w_{2} = 0.5, w_{3} = 5, w_{4} = 1, w_{5} = 0.25, w_{6} = 0.75\}$$

</div>

Has an $L_{2}$ regularization term of 26.915:

<div class="math-jax-block">

$= w_{1}^2 + w_{2}^2 + w_{3}^2 + w_{4}^2 + w_{5}^2 + w_{6}^2$

</div>

<div class="math-jax-block">

$= 0.2^2 + 0.5^2 + 5^2 + 1^2 + 0.25^2 + 0.75^2$

</div>

<div class="math-jax-block">

$= 0.04 + 0.24 + 25 + 1 + 0.0625 + 0.5625$

</div>

<div class="math-jax-block">

$= 26.915$

</div>

But $w_{3}$, with a squared value of 25, contributes nearly all the complexity. The sum of the squares of all five other weights adds just 1.915 to the $L_{2}$ regularization term.

## Lambda

Model developers tune the overall impact of the regularization term by multiplying its value by a scalar known as **lambda** (also called the **regularization rate**). That is, model devvelopers aim to do the following:

<div class="math-jax-block">

$$\mathrm{minimize(Loss(Data \vert Model) + \lambda \ complexity(Model))}$$

</div>

Performing $L_{2}$ regularization has the following effect on the model:

  * Encourages weight values toward 0 (but not exactly 0).
  * Encourages the mean of the weights toward 0, with a normal (bell-shaped or Gaussian) distribution.

Increasing the lambda value strengthens the regularization effect. For example, the histogram of weights for a high value of lambda might look as shown in Figure 2.

<div align='center'>
  <img src='https://developers.google.com/static/machine-learning/crash-course/images/HighLambda.svg' height='400px' />

  <strong>Figure 2. Histogram of weights.</strong>
</div>

Lowering the value of lambda tends to yield a flatter histogram, as shown in Figure 3.

<div align='center'>
  <img src='https://developers.google.com/static/machine-learning/crash-course/images/LowLambda.svg' />

  <strong>Figure 3. Histogram of weights produced by a lower lambda value.</strong>
</div>

When choosing a lambda value, the goal is to strike the right balance between simplicity and training-data fit:

* If your lambda value is too high, your model will be simple, but you run the risk of underfitting your data. Your model won't learn enough about the training data to make useful predictions.
* If your lambda value is too low, your model will be more complex, and you run the risk of overfitting your data. Your model will learn too much about the particularities of the training data, and won't be able to generalize to new data.

The ideal value of lambda produces a model that generalizes well to new, previously unseen data. Unfortunately, that ideal value of lambda is data-dependent, so you'll need to do some tuning.

### L2 regularization and Learning rate

There's a close connection between learning rate and lambda. Strong L2 regularization values tend to drive feature weights closer to 0. Lower learning rates (with early stopping) often produce the same effect because the steps away from 0 aren't as large. Consequently, tweaking learning rate and lambda simultaneously may have confounding effects.

**Early stopping** means ending training before the model fully reaches convergence. In practice, we often end up with some amount of implicit early stopping when training in an [online](https://developers.google.com/machine-learning/crash-course/production-ml-systems) (continuous) fashion. That is, some new trends just haven't had enough data yet to converge.

As noted, the effects from changes to regularization parameters can be confounded with the effects from changes in learning rate or number of iterations. One useful practice (when training across a fixed batch of data) is to give yourself a high enough number of iterations that early stopping doesn't play into things.

# Logistic Regression

Instead of predicting *exactly* 0 or 1, **logistic regression** generates a probability—a value between 0 and 1, exclusive. For example, consider a logistic regression model for spam detection. If the model infers a value of 0.932 on a particular email message, it implies a 93.2% probability that the email message is spam. More precisely, it means that in the limit of *infinite* training examples, the set of examples for which the model predicts 0.932 will actually be spam 93.2% of the time and the remaining 6.8% will not.

## Calculating a Probability

Many problems require a probability estimate as output. Logistic regression is an extremely efficient mechanism for calculating probabilities. Practically speaking, you can use the returned probability in either of the following two ways:

* "As is"
* Converted to a binary category.

Let's consider how we might use the probability "as is." Suppose we create a logistic regression model to predict the probability that a dog will bark during the middle of the night. We'll call that probability:

$$\mathrm{p}(\mathrm{bark} \vert \mathrm{night})$$

If the logistic regression model predicts $\mathrm{p}(\mathrm{bark} \vert \mathrm{night}) = 0.05$, then over a year, the dog's owners should be startled awake approximately 18 times:

$$\mathrm{startled} = \mathrm{p}(\mathrm{bark} \vert \mathrm{night}) \cdot \mathrm{nights}$$
$$= 0.05 \cdot 365$$
$$= 18$$

In many cases, you'll map the logistic regression output into the solution to a binary classification problem, in which the goal is to correctly predict one of two possible labels (e.g., "spam" or "not spam").

You might be wondering how a logistic regression model can ensure output that always falls between 0 and 1. As it happens, a **sigmoid function**, defined as follows, produces output having those same characteristics:

$$y = \dfrac{1}{1 + e^{-z}}$$

The sigmoid function yields the following plot:

<div align='center'>
  <img src='https://developers.google.com/machine-learning/crash-course/images/SigmoidFunction.png' />

  <strong>Figure 1: Sigmoid function.</strong>
</div>

If $z$ represents the output of the linear layer of a model trained with logistic regression, then $sigmoid(z)$ will yield a value (a probability) between 0 and 1. In mathematical terms:

$$y^ \prime = \dfrac{1}{1 + e^{-z}}$$

where:

* $y^ \prime$ is the output of the logistic regression model for a particular example.

For $z$:

<div class="math-jax-block">

$$z = b + w_{1}x_{1} + w_{2}x_{2} + ... + w_{N}x_{N}$$

</div>

where:

  * The $w$ values are the model's learned weights, and $b$ is the bias.
  * The $x$ values are the feature values for a particular example.

Note that $z$ is also referred to as the *log-odds* because the inverse of the sigmoid states that $z$ can be defined as the log of the probability of the 1 label (eg., "dog barks") divided by the probability of the 0 label (eg., "dog doesn't bark"):

$$z = \mathrm{log} \left( \dfrac{y}{1- y} \right)$$

Here is the sigmoid function with ML labels:

<div align='center'>
  <img src='https://developers.google.com/static/machine-learning/crash-course/images/LogisticRegressionOutput.svg' />

  <strong>Figure 2: Logistic regression output.</strong>
</div>

## Loss and Regularization

### Loss function for Logistic Regression

The loss function for linear regression is squared loss. The loss function for logistic regression is **Log Loss**, which is defined as follows:

<div class="math-jax-block">

$$\mathrm{Log\ Loss} = \sum_{(x,y) \in D} -ylog(y^ \prime) - (1 - y)log(1 - y^ \prime)$$

</div>

where:

* $(x,y) \in D$ is the data set containing many labeled examples, which are $(x,y)$ pairs.
* $y$ is the label in a labeled example. Since this is logistic regression, every value of $y$ must either be 0 or 1.
* $y^ \prime$ is the predicted value (somewhere between 0 and 1), given the set of features in $x$.

### Regularization in Logistic Regression

[Regularization](https://developers.google.com/machine-learning/crash-course/regularization-for-simplicity/video-lecture) is extremely important in logistic regression modeling. Without regularization, the asymptotic nature of logistic regression would keep driving loss towards 0 in high dimensions. Consequently, most logistic regression models use one of the following two strategies to dampen model complexity:

* $L_{2}$ regularization
* Early stopping, that is, limiting the number of training steps or the learning rate.

Imagine that you assign a unique id to each example, and map each id to its own feature. If you don't specify a regularization function, the model will become completely overfit. That's because the model would try to drive loss to zero on all examples and never get there, driving the weights for each indicator feature to +infinity or -infinity. This can happen in high dimensional data with feature crosses, when there’s a huge mass of rare crosses that happen only on one example each.

Fortunately, using L2 or early stopping will prevent this problem.

# Classification

## Thresholding

Logistic regression returns a probability. You can use the returned probability "as is" (for example, the probability that the user will click on this ad is 0.00023) or convert the returned probability to a binary value (for example, this email is spam).

A logistic regression model that returns 0.9995 for a particular email message is predicting that it is very likely to be spam. Conversely, another email message with a prediction score of 0.0003 on that same logistic regression model is very likely not spam. However, what about an email message with a prediction score of 0.6? In order to map a logistic regression value to a binary category, you must define a **classification threshold** (also called the **decision threshold**). A value above that threshold indicates "spam"; a value below indicates "not spam." It is tempting to assume that the classification threshold should always be 0.5, but thresholds are problem-dependent, and are therefore values that you must tune.

## True vs. False and Positive vs. Negative

In this section, we'll define the primary building blocks of the metrics we'll use to evaluate classification models. But first, a fable:

**An Aesop's Fable: The Boy Who Cried Wolf (compressed)**
  
A shepherd boy gets bored tending the town's flock. To have some fun, he cries out, "Wolf!" even though no wolf is in sight. The villagers run to protect the flock, but then get really mad when they realize the boy was playing a joke on them.

[Iterate previous paragraph N times.]

One night, the shepherd boy sees a real wolf approaching the flock and calls out, "Wolf!" The villagers refuse to be fooled again and stay in their houses. The hungry wolf turns the flock into lamb chops. The town goes hungry. Panic ensues.

Let's make the following definitions:

* "Wolf" is a **positive class**.
* "No wolf" is a **negative class**.

We can summarize our "wolf-prediction" model using a 2x2 [confusion matrix](https://developers.google.com/machine-learning/glossary#confusion_matrix) that depicts all four possible outcomes:

* **True Positive (TP):**
  * Reality: A wolf threatened.
  * Shepherd said: "Wolf."
  * Outcome: Shepherd is a hero.

* **False Positive (FP):**
  * Reality: No wolf threatened.
  * Shepherd said: "Wolf."
  * Outcome: Villagers are angry at shepherd for waking them up.

* **True Negative (TN):**
  * Reality: No wolf threatened.
  * Shepherd said: "No wolf."
  * Outcome: Everyone is fine.

* **False Negative (FN):**
  * Reality: A wolf threatened.
  * Shepherd said: "No wolf."
  * Outcome: The wolf ate all the sheep.

A **true positive** is an outcome where the model correctly predicts the positive class. Similarly, a **true negative** is an outcome where the model correctly predicts the negative class.

A **false positive** is an outcome where the model incorrectly predicts the positive class. And a **false negative** is an outcome where the model incorrectly predicts the negative class.

## Accuracy

Accuracy is one metric for evaluating classification models. Informally, **accuracy** is the fraction of predictions our model got right. Formally, accuracy has the following definition:

<div class="math-jax-block">

$\mathrm{Accuracy} = \dfrac{\mathrm{Number\ of\ correct\ predictions}}{\mathrm{Total\ number\ of\ predictions}}$

</div>

For binary classification, accuracy can also be calculated in terms of positives and negatives as follows:

<div class="math-jax-block">

$\mathrm{Accuracy} = \dfrac{TP + TN}{TP + TN + FP + FN}$

</div>

Where *TP* = True Positives, *TN* = True Negatives, *FP* = False Positives, and *FN* = False Negatives.

Let's try calculating accuracy for the following model that classified 100 tumors as [malignant](https://wikipedia.org/wiki/Malignancy) (the positive class) or [benign](https://wikipedia.org/wiki/Benign_tumor) (the negative class):

* **True Positive (TP):**
  * Reality: Malignant
  * ML model predicted: Malignant
  * **Number of TP results**: 1

* **False Positive (FP):**
  * Reality: Benign
  * ML model predicted: Malignant
  * **Number of TP results**: 1

* **True Negative (TN):**
  * Reality: Benign
  * ML model predicted: Benign
  * **Number of TP results**: 90

* **False Negative (FN):**
  * Reality: Malignant
  * ML model predicted: Benign
  * **Number of TP results**: 8

<div class="math-jax-block">

$\mathrm{Accuracy} = \dfrac{TP + TN}{TP + TN + FP + FN} = \dfrac{1 + 90}{1 + 90 + 1 + 8} = 0.91$

</div>

---



Accuracy comes out to 0.91, or 91% (91 correct predictions out of 100 total examples). That means our tumor classifier is doing a great job of identifying malignancies, right?

Actually, let's do a closer analysis of positives and negatives to gain more insight into our model's performance.

Of the 100 tumor examples, 91 are benign (90 TNs and 1 FP) and 9 are malignant (1 TP and 8 FNs).

Of the 91 benign tumors, the model correctly identifies 90 as benign. That's good. However, of the 9 malignant tumors, the model only correctly identifies 1 as malignant—a terrible outcome, as 8 out of 9 malignancies go undiagnosed!

While 91% accuracy may seem good at first glance, another tumor-classifier model that always predicts benign would achieve the exact same accuracy (91/100 correct predictions) on our examples. In other words, our model is no better than one that has zero predictive ability to distinguish malignant tumors from benign tumors.

Accuracy alone doesn't tell the full story when you're working with a **class-imbalanced data set**, like this one, where there is a significant disparity between the number of positive and negative labels.

## Precision and Recall

### Precision

Precision attempts to answer the following question:

```
What proportion of positive identifications was actually correct?
```

Precision is defined as follows:

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FP}$

</div>

Let's calculate precision for our ML model from the previous section that analyzes tumors:

* **True Positives (TPs): 1**
* **False Positives (FPs): 1**
* **False Negatives (FNs): 8**
* **True Negatives (TNs): 90**

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FP} = \dfrac{1}{1 + 1} = 0.5$

</div>

Our model has a precision of 0.5—in other words, when it predicts a tumor is malignant, it is correct 50% of the time.

### Recall

**Recall** attempts to answer the following question:

```
What proportion of actual positives was identified correctly?
```

Mathematically, recall is defined as follows:

$\mathrm{Recall} = \dfrac{TP}{TP + FN}$

Let's calculate recall for our tumor classifier:

* **True Positives (TPs): 1**
* **False Positives (FPs): 1**
* **False Negatives (FNs): 8**
* **True Negatives (TNs): 90**

<div class="math-jax-block">

$\mathrm{Recall} = \dfrac{TP}{TP + FN} = \dfrac{1}{1 + 8} = 0.11$

</div>

Our model has a recall of 0.11—in other words, it correctly identifies 11% of all malignant tumors.

### Precision and Recall: A Tug of War

To fully evaluate the effectiveness of a model, you must examine both precision and recall. Unfortunately, precision and recall are often in tension. That is, improving precision typically reduces recall and vice versa. Explore this notion by looking at the following figure, which shows 30 predictions made by an email classification model. Those to the right of the classification threshold are classified as "spam", while those to the left are classified as "not spam."

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/PrecisionVsRecallBase.svg" />

<strong>Figure 1. Classifying email messages as spam or not spam.</strong>

</div>

Let's calculate precision and recall based on the results shown in Figure 1:

* **True Positives (TP): 8**
* **False Positives (FP): 2**
* **False Negatives (FN): 3**
* **True Negatives (TN): 17**

Precision measures the percentage of **emails flagged as spam** that were correctly classified—that is, the percentage of dots to the right of the threshold line that are green in Figure 1:

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FP} = \dfrac{8}{8 + 2} = 0.8$

</div>

Recall measures the percentage of **actual spam emails** that were correctly classified—that is, the percentage of green dots that are to the right of the threshold line in Figure 1:

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FN} = \dfrac{8}{8 + 3} = 0.73$

</div>

Figure 2 illustrates the effect of increasing the classification threshold.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/PrecisionVsRecallRaiseThreshold.svg" />

<strong>Figure 2. Increasing classification threshold.</strong>

</div>

The number of false positives decreases, but false negatives increase. As a result, precision increases, while recall decreases:

* **True Positives (TP): 7**
* **False Positives (FP): 1**
* **False Negatives (FN): 4**
* **True Negatives (TN): 18**

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FP} = \dfrac{7}{7 + 1} = 0.88$

</div>

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FN} = \dfrac{7}{7 + 4} = 0.64$

</div>

Conversely, Figure 3 illustrates the effect of decreasing the classification threshold (from its original position in Figure 1).

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/PrecisionVsRecallLowerThreshold.svg" />

<strong>Figure 3. Decreasing classification threshold.</strong>

</div>

False positives increase, and false negatives decrease. As a result, this time, precision decreases and recall increases:

* **True Positives (TP): 9**
* **False Positives (FP): 3**
* **False Negatives (FN): 2**
* **True Negatives (TN): 16**

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FP} = \dfrac{9}{9 + 3} = 0.75$

</div>

<div class="math-jax-block">

$\mathrm{Precision} = \dfrac{TP}{TP + FN} = \dfrac{9}{9 + 2} = 0.82$

</div>

Various metrics have been developed that rely on both precision and recall. For example, see [F1 score](https://wikipedia.org/wiki/F1_score).

## ROC Curve and AUC

### ROC Curve

An **ROC curve (receiver operating characteristic curve)** is a graph showing the performance of a classification model at all classification thresholds. This curve plots two parameters:

* True positive rate
* False positive rate

**True Positive Rate (TPR)** is a synonym for recall and is therefore defined as follows:

$\mathrm{TPR} = \dfrac{TP}{TP + FN}$

**False Positive Rate (FPR)** is defined as follows:

$\mathrm{FPR} = \dfrac{FP}{FP + TN}$

An ROC curve plots TPR vs. FPR at different classification thresholds. Lowering the classification threshold classifies more items as positive, thus increasing both False Positives and True Positives. The following figure shows a typical ROC curve.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/ROCCurve.svg" />

<strong>Figure 4. TP vs. FP rate at different classification thresholds.</strong>

</div>

To compute the points in an ROC curve, we could evaluate a logistic regression model many times with different classification thresholds, but this would be inefficient. Fortunately, there's an efficient, sorting-based algorithm that can provide this information for us, called AUC.

### AUC: Area Under the ROC Curve

**AUC** stands for "Area under the ROC Curve." That is, AUC measures the entire two-dimensional area underneath the entire ROC curve (think integral calculus) from (0,0) to (1,1).

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/AUC.svg" />

<strong>Figure 5. AUC (Area under the ROC Curve).</strong>

</div>

AUC provides an aggregate measure of performance across all possible classification thresholds. One way of interpreting AUC is as the probability that the model ranks a random positive example more highly than a random negative example. For example, given the following examples, which are arranged from left to right in ascending order of logistic regression predictions:

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/AUCPredictionsRanked.svg" />

<strong>Figure 6. Predictions ranked in ascending order of logistic regression score.</strong>

</div>

AUC represents the probability that a random positive (green) example is positioned to the right of a random negative (red) example.

AUC ranges in value from 0 to 1. A model whose predictions are 100% wrong has an AUC of 0.0; one whose predictions are 100% correct has an AUC of 1.0.

AUC is desirable for the following two reasons:

* AUC is **scale-invariant**. It measures how well predictions are ranked, rather than their absolute values.
* AUC is **classification-threshold-invariant**. It measures the quality of the model's predictions irrespective of what classification threshold is chosen.

However, both these reasons come with caveats, which may limit the usefulness of AUC in certain use cases:

* **Scale invariance is not always desirable**. For example, sometimes we really do need well calibrated probability outputs, and AUC won’t tell us about that.

* **Classification-threshold invariance is not always desirable**. In cases where there are wide disparities in the cost of false negatives vs. false positives, it may be critical to minimize one type of classification error. For example, when doing email spam detection, you likely want to prioritize minimizing false positives (even if that results in a significant increase of false negatives). AUC isn't a useful metric for this type of optimization.

## Prediction Bias

Logistic regression predictions should be unbiased. That is:

```
"average of predictions" should ≈ "average of observations"
```

**Prediction bias** is a quantity that measures how far apart those two averages are. That is:

<div class="math-jax-block">

$\mathrm{prediction\ bias} = \mathrm{average\ of\ predictions} - \mathrm{average\ of\ labels\ in\ data\ set}$

</div>

<small>**Note:** "Prediction bias" is a different quantity than [bias](https://developers.google.com/machine-learning/crash-course/descending-into-ml) (the b in wx + b).</small>

A significant nonzero prediction bias tells you there is a bug somewhere in your model, as it indicates that the model is wrong about how frequently positive labels occur.

For example, let's say we know that on average, 1% of all emails are spam. If we don't know anything at all about a given email, we should predict that it's 1% likely to be spam. Similarly, a good spam model should predict on average that emails are 1% likely to be spam. (In other words, if we average the predicted likelihoods of each individual email being spam, the result should be 1%.) If instead, the model's average prediction is 20% likelihood of being spam, we can conclude that it exhibits prediction bias.

Possible root causes of prediction bias are:

* Incomplete feature set
* Noisy data set
* Buggy pipeline
* Biased training sample
* Overly strong regularization

You might be tempted to correct prediction bias by post-processing the learned model—that is, by adding a **calibration layer** that adjusts your model's output to reduce the prediction bias. For example, if your model has +3% bias, you could add a calibration layer that lowers the mean prediction by 3%. However, adding a calibration layer is a bad idea for the following reasons:

* You're fixing the symptom rather than the cause.
* You've built a more brittle system that you must now keep up to date.

If possible, avoid calibration layers. Projects that use calibration layers tend to become reliant on them—using calibration layers to fix all their model's sins. Ultimately, maintaining the calibration layers can become a nightmare.

<small>**Note:** A good model will usually have near-zero bias. That said, a low prediction bias does not prove that your model is good. A really terrible model could have a zero prediction bias. For example, a model that just predicts the mean value for all examples would be a bad model, despite having zero bias.</small>

### Bucketing and Prediction Bias

Logistic regression predicts a value between 0 and 1. However, all labeled examples are either exactly 0 (meaning, for example, "not spam") or exactly 1 (meaning, for example, "spam"). Therefore, when examining prediction bias, you cannot accurately determine the prediction bias based on only one example; you must examine the prediction bias on a "bucket" of examples. That is, prediction bias for logistic regression only makes sense when grouping enough examples together to be able to compare a predicted value (for example, 0.392) to observed values (for example, 0.394).

You can form buckets in the following ways:

* Linearly breaking up the target predictions.
* Forming quantiles.

Consider the following calibration plot from a particular model. Each dot represents a bucket of 1,000 values. The axes have the following meanings:

* The x-axis represents the average of values the model predicted for that bucket.
* The y-axis represents the actual average of values in the data set for that bucket.

Both axes are logarithmic scales.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/BucketingBias.svg" />

<strong>Figure 8. Prediction bias curve (logarithmic scales)</strong>

</div>

Why are the predictions so poor for only part of the model? Here are a few possibilities:

* The training set doesn't adequately represent certain subsets of the data space.
* Some subsets of the data set are noisier than others.
* The model is overly [regularized](https://developers.google.com/machine-learning/crash-course/regularization-for-simplicity/video-lecture). (Consider reducing the value of [lambda](https://developers.google.com/machine-learning/glossary#lambda).)

# Regularization for Sparsity

## L₁ Regularization

Sparse vectors often contain many dimensions. Creating a [feature cross](https://developers.google.com/machine-learning/crash-course/feature-crosses/video-lecture) results in even more dimensions. Given such high-dimensional feature vectors, model size may become huge and require huge amounts of RAM.

In a high-dimensional sparse vector, it would be nice to encourage weights to drop to exactly 0 where possible. A weight of exactly 0 essentially removes the corresponding feature from the model. Zeroing out features will save RAM and may reduce noise in the model.

For example, consider a housing data set that covers not just California but the entire globe. Bucketing global latitude at the minute level (60 minutes per degree) gives about 10,000 dimensions in a sparse encoding; global longitude at the minute level gives about 20,000 dimensions. A feature cross of these two features would result in roughly 200,000,000 dimensions. Many of those 200,000,000 dimensions represent areas of such limited residence (for example, the middle of the ocean) that it would be difficult to use that data to generalize effectively. It would be silly to pay the RAM cost of storing these unneeded dimensions. Therefore, it would be nice to encourage the weights for the meaningless dimensions to drop to exactly 0, which would allow us to avoid paying for the storage cost of these model coefficients at inference time.

We might be able to encode this idea into the optimization problem done at training time, by adding an appropriately chosen regularization term.

Would $L_{2}$ regularization accomplish this task? Unfortunately not. $L_{2}$ regularization encourages weights to be small, but doesn't force them to exactly 0.0.

An alternative idea would be to try and create a regularization term that penalizes the count of non-zero coefficient values in a model. Increasing this count would only be justified if there was a sufficient gain in the model's ability to fit the data. Unfortunately, while this count-based approach is intuitively appealing, it would turn our convex optimization problem into a non-convex optimization problem. So this idea, known as $L_{0}$ regularization isn't something we can use effectively in practice.

However, there is a regularization term called $L_{1}$ regularization that serves as an approximation to $L_{0}$, but has the advantage of being convex and thus efficient to compute. So we can use $L_{1}$ regularization to encourage many of the uninformative coefficients in our model to be exactly 0, and thus reap RAM savings at inference time.

### $L_{1}$ vs. $L_{2}$ regularization

$L_{2}$ and $L_{1}$ penalize weights differently:

* $L_{2}$ penalizes $weight^2$.
* $L_{1}$ penalizes $|weight|$.

Consequently, $L_{2}$ and $L_{1}$ have different derivatives:

* The derivative of $L_{2}$ is 2 * weight.
* The derivative of $L_{1}$ is k (a constant, whose value is independent of weight).

You can think of the derivative of $L_{2}$ as a force that removes x% of the weight every time. As [Zeno](https://en.wikipedia.org/wiki/Zeno's_paradoxes#Dichotomy_paradox) knew, even if you remove x percent of a number billions of times, the diminished number will still never quite reach zero. (Zeno was less familiar with floating-point precision limitations, which could possibly produce exactly zero.) At any rate, $L_{2}$ does not normally drive weights to zero.

You can think of the derivative of $L_{1}$ as a force that subtracts some constant from the weight every time. However, thanks to absolute values, $L_{1}$ has a discontinuity at 0, which causes subtraction results that cross 0 to become zeroed out. For example, if subtraction would have forced a weight from +0.1 to -0.2, $L_{1}$ will set the weight to exactly 0. Eureka, $L_{1}$ zeroed out the weight.

$L_{1}$ regularization—penalizing the absolute value of all the weights—turns out to be quite efficient for wide models.

Note that this description is true for a one-dimensional model.

To compare the results of $L_{1}$ and $L_{2}$ regularization on a network of weights, click the play button on the [playground](https://developers.google.com/machine-learning/crash-course/regularization-for-sparsity/l1-regularization#l1-vs.-l2-regularization.).

# Neural Networks

Neural networks are a more sophisticated version of feature crosses. In essence, neural networks learn the appropriate feature crosses for you.

## Structure

If you recall from the Feature Crosses unit, the following classification problem is nonlinear:

<div align="center">

<img src="https://developers.google.com/machine-learning/crash-course/images/FeatureCrosses1.png" />

<strong>Figure 1. Nonlinear classification problem.</strong>

</div>

"Nonlinear" means that you can't accurately predict a label with a model of the form $b + w_{1}x_{1} + w_{2}x_{2}$ In other words, the "decision surface" is not a line. Previously, we looked at feature crosses as one possible approach to modeling nonlinear problems.

Now consider the following data set:

<div align="center">

<img src="https://developers.google.com/machine-learning/crash-course/images/NonLinearSpiral.png" />

<strong>Figure 2. A more difficult nonlinear classification problem.</strong>

</div>

The data set shown in Figure 2 can't be solved with a linear model.

To see how neural networks might help with nonlinear problems, let's start by representing a linear model as a graph:

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/linear_net.svg" />

<strong>Figure 3. Linear model as graph.</strong>

</div>

Each blue circle represents an input feature, and the green circle represents the weighted sum of the inputs.

### The Linear Unit

So let's begin with the fundamental component of a neural network: the individual neuron. As a diagram, a **neuron** (or **unit**) with one input looks like:

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/mfOlDR6.png" />

<strong><i>The Linear Unit:</i> y = wx + b</strong>

</div>

The input is $x$. Its connection to the neuron has a **weight** which is $w$. Whenever a value flows through a connection, you multiply the value by the connection's weight. For the input $x$, what reaches the neuron is $w * x$. A neural network "learns" by modifying its weights.

The $b$ is a special kind of weight we call it **bias**. The bias doesn't have any input data associated with it; instead, we put a $1$ in the diagram so that the value that reaches the neuron is just $b$ (since $1 * b = b$). The bias enables the neuron to modify the output independently of its inputs.

The $y$ is the value the neuron ultimately outputs. To get the output, the neuron sums up all the values it receives through its connections. This neuron's activation is $y = w * x + b$, or as a formula $y=wx+b$.

### Multiple Inputs

In the previous section we saw how can we handle a single input using *The Linear Unit*, but what if we wanted to expand our model to include more inputs? That's easy enough. We can just add more input connections to the neuron, one for each additional feature. To find the output, we would multiply each input to its connection weight and then add them all together.

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/vyXSnlZ.png" />

<strong>A linear unit with three inputs.</strong>

</div>

The formula for this neuron would be $y=w0x0+w1x1+w2x2+b$. A linear unit with two inputs will fit a plane, and a unit with more inputs than that will fit a hyperplane.

### Layers

Neural networks typically organize their neurons into **layers**. When we collect together linear units having a common set of inputs we get a **dense** layer.

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/2MA4iMV.png" />

<strong>A dense layer of two linear units receiving two inputs and a bias.</strong>

</div>

You could think of each layer in a neural network as performing some kind of relatively simple transformation. Through a deep stack of layers, a neural network can transform its inputs in more and more complex ways. In a well-trained neural network, each layer is a transformation getting us a little bit closer to a solution.

### Hidden Layers

In the model represented by the following graph, we've added a "hidden layer" of intermediary values. Each yellow node in the hidden layer is a weighted sum of the blue input node values. The output is a weighted sum of the yellow nodes.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/1hidden.svg" height='400px' />
<strong>Figure 4. Graph of two-layer model.</strong>

</div>

Is this model linear? Yes—its output is still a linear combination of its inputs.

In the model represented by the following graph, we've added a second hidden layer of weighted sums.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/2hidden.svg" height="400px" />

<strong>Figure 5. Graph of three-layer model.</strong>

</div>

Is this model still linear? Yes, it is. When you express the output as a function of the input and simplify, you get just another weighted sum of the inputs. This sum won't effectively model the nonlinear problem in Figure 2.

### Activation Functions

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/OLSUEYT.png" />

<i>Without activation functions, neural networks can only learn linear relationships. In order to fit curves, we'll need to use activation functions.</i>

</div>

To model a nonlinear problem, we can directly introduce a nonlinearity. We can pipe each hidden layer node through a nonlinear function.

In the model represented by the following graph, the value of each node in Hidden Layer 1 is transformed by a nonlinear function before being passed on to the weighted sums of the next layer. This nonlinear function is called the activation function.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/activation.svg" />

<strong>Figure 6. Graph of three-layer model with activation function.</strong>

</div>

An **activation function** is simply some function we apply to each of a layer's outputs (its activations). The most common is the rectifier function  $max(0,x)$.

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/aeIyAlF.png" />

</div>

The rectifier function has a graph that's a line with the negative part "rectified" to zero. Applying the function to the outputs of a neuron will put a bend in the data, moving us away from simple lines.

When we attach the rectifier to a linear unit, we get a **rectified linear unit** or **ReLU**. (For this reason, it's common to call the rectifier function the "ReLU function".) Applying a ReLU activation to a linear unit means the output becomes $max(0, w * x + b)$, which we might draw in a diagram like:

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/eFry7Yu.png" />

<i>A rectified linear unit.</i>

</div>

Now that we've added an activation function, adding layers has more impact. Stacking nonlinearities on nonlinearities lets us model very complicated relationships between the inputs and the predicted outputs. In brief, each layer is effectively learning a more complex, higher-level function over the raw inputs. If you'd like to develop more intuition on how this works, see [Chris Olah's excellent blog post](http://colah.github.io/posts/2014-03-NN-Manifolds-Topology/).

### Common Activation Functions

The following **sigmoid** activation function converts the weighted sum to a value between 0 and 1.

$$F(x)=\dfrac{1}{1 + e^{-x}}$$

Here's a plot:

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/sigmoid.svg" />

<strong>Figure 7. Sigmoid activation function.</strong>

</div>

The following **rectified linear unit** activation function (or **ReLU**, for short) often works a little better than a smooth function like the sigmoid, while also being significantly easier to compute.

$$F(x) = max(0, x)$$

The superiority of ReLU is based on empirical findings, probably driven by ReLU having a more useful range of responsiveness. A sigmoid's responsiveness falls off relatively quickly on both sides.

<div align="center">

<img src="https://developers.google.com/static/machine-learning/crash-course/images/relu.svg" />

<strong>Figure 8. ReLU activation function.</strong>

</div>

In fact, any mathematical function can serve as an activation function. Suppose that $\sigma$ represents our activation function (Relu, Sigmoid, or whatever). Consequently, the value of a node in the network is given by the following formula:

$$\sigma(w.x + b)$$

TensorFlow provides out-of-the-box support for many activation functions. You can find these activation functions within TensorFlow's [list of wrappers for primitive neural network operations](https://www.tensorflow.org/api_docs/python/tf/nn). That said, we still recommend starting with ReLU.

### Stacking Dense Layers

Now that we have some nonlinearity, let's see how we can stack layers to get complex data transformations.

<div align='center'>

<img src="https://storage.googleapis.com/kaggle-media/learn/images/Y5iwFQZ.png" />

<i>A stack of dense layers makes a "fully-connected" network.</i>

</div>

The layers before the output layer are sometimes called **hidden** since we never see their outputs directly.

Now, notice that the final (output) layer is a linear unit (meaning, no activation function). That makes this network appropriate to a regression task, where we are trying to predict some arbitrary numeric value. Other tasks (like classification) might require an activation function on the output.

### Summary

Now our model has all the standard components of what people usually mean when they say "neural network":

* A set of nodes, analogous to neurons, organized in layers.
* A set of weights representing the connections between each neural network layer and the layer beneath it. The layer beneath may be another neural network layer, or some other kind of layer.
* A set of biases, one for each node.
* An activation function that transforms the output of each node in a layer. Different layers may have different activation functions.

# Training Neural Networks

**Backpropagation** is the most common training algorithm for neural networks. It makes gradient descent feasible for multi-layer neural networks.

## Best Practices

This section explains backpropagation's failure cases and the most common way to regularize a neural network.

### Failure Cases

There are a number of common ways for backpropagation to go wrong.

#### Vanishing Gradients

The gradients for the lower layers (closer to the input) can become very small. In deep networks, computing these gradients can involve taking the product of many small terms.

When the gradients vanish toward 0 for the lower layers, these layers train very slowly, or not at all.

The ReLU activation function can help prevent vanishing gradients.

#### Exploding Gradients

If the weights in a network are very large, then the gradients for the lower layers involve products of many large terms. In this case you can have exploding gradients: gradients that get too large to converge.

Batch normalization can help prevent exploding gradients, as can lowering the learning rate.

#### Dead ReLU Units

Once the weighted sum for a ReLU unit falls below 0, the ReLU unit can get stuck. It outputs 0 activation, contributing nothing to the network's output, and gradients can no longer flow through it during backpropagation. With a source of gradients cut off, the input to the ReLU may not ever change enough to bring the weighted sum back above 0.

Lowering the learning rate can help keep ReLU units from dying.

### Dropout Regularization

Yet another form of regularization, called **Dropout**, is useful for neural networks. It works by randomly "dropping out" unit activations in a network for a single gradient step. The more you drop out, the stronger the regularization:

* 0.0 = No dropout regularization.
* 1.0 = Drop out everything. The model learns nothing.
* Values between 0.0 and 1.0 = More useful.

<div align="center">

<img src="https://storage.googleapis.com/kaggle-media/learn/images/a86utxY.gif" />

<i>Here, 50% dropout has been added between the two hidden layers.</i>

</div>

### Batch Normalization (batchnorm)

With neural networks, it's generally a good idea to put all of your data on a common scale. The reason is that SGD will shift the network weights in proportion to how large an activation the data produces. Features that tend to produce activations of very different sizes can make for unstable training behavior.

Now, if it's good to normalize the data before it goes into the network, maybe also normalizing inside the network would be better! In fact, we have a special kind of layer that can do this, the **batch normalization layer**. A batch normalization layer looks at each batch as it comes in, first normalizing the batch with its own mean and standard deviation, and then also putting the data on a new scale with two trainable rescaling parameters. Batchnorm, in effect, performs a kind of coordinated rescaling of its inputs.

Most often, batchnorm is added as an aid to the optimization process (though it can sometimes also help prediction performance). Models with batchnorm tend to need fewer epochs to complete training. Moreover, batchnorm can also fix various problems that can cause the training to get "stuck". Consider adding batch normalization to your models, especially if you're having trouble during training.